In [12]:
# CELL 1: Setup & Imports
import re
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, GlobalMaxPooling1D,
    Dense, Dropout, Lambda, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


In [13]:
# CELL 2: Text Cleaning & Preprocessing Pipeline
def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class TextPreprocessor:
    def __init__(self, max_vocab_size=15000, max_sequence_length=140):
        self.max_vocab_size = max_vocab_size
        self.max_sequence_length = max_sequence_length
        self.tokenizer = Tokenizer(num_words=self.max_vocab_size, oov_token="<OOV>")

    def fit(self, texts_a, texts_b):
        cleaned_a = [clean_text(t) for t in texts_a]
        cleaned_b = [clean_text(t) for t in texts_b]
        self.tokenizer.fit_on_texts(cleaned_a + cleaned_b)
        print(f"Tokenizer fitted. Actual vocabulary size: {len(self.tokenizer.word_index)}")
        return self

    def transform(self, texts):
        cleaned_texts = [clean_text(t) for t in texts]
        sequences = self.tokenizer.texts_to_sequences(cleaned_texts)
        return pad_sequences(
            sequences,
            maxlen=self.max_sequence_length,
            padding='post',
            truncating='post'
        )

In [14]:
# CELL 3: Model Architecture
@tf.keras.utils.register_keras_serializable(name="AbsoluteDifference")
def absolute_difference(tensors):
    return tf.abs(tensors[0] - tensors[1])

@tf.keras.utils.register_keras_serializable(name="ElementwiseProduct")
def elementwise_product(tensors):
    return tensors[0] * tensors[1]

def build_resume_jd_siamese(max_len=140, vocab_size=15000):
    # 1. Inputs (Resume and Job Description)
    input_resume = Input(shape=(max_len,), dtype='int32', name='input_resume')
    input_jd = Input(shape=(max_len,), dtype='int32', name='input_jd')

    # 2. Shared Layers
    shared_embedding = Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True, name='shared_embedding')
    shared_bilstm = Bidirectional(LSTM(64, return_sequences=True), name='shared_bilstm')
    shared_pooling = GlobalMaxPooling1D(name='shared_global_max_pooling')

    # 3. Process Both Inputs
    emb_resume = shared_embedding(input_resume)
    lstm_resume = shared_bilstm(emb_resume)
    u = shared_pooling(lstm_resume)

    emb_jd = shared_embedding(input_jd)
    lstm_jd = shared_bilstm(emb_jd)
    v = shared_pooling(lstm_jd)

    # 4. Interactive Comparison (The u, v, |u-v|, u*v logic)
    diff_layer = Lambda(absolute_difference, name='absolute_difference')([u, v])
    prod_layer = Lambda(elementwise_product, name='elementwise_product')([u, v])
    merged_features = Concatenate(name='feature_concatenation')([u, v, diff_layer, prod_layer])

    # 5. Classification Head (0=Weak, 1=Medium, 2=Strong)
    x = Dense(128, activation='relu')(merged_features)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(3, activation='softmax', name='match_score')(x)

    return Model(inputs=[input_resume, input_jd], outputs=output, name='Siamese_Resume_JD_Matcher')

# Initialize and view the architecture
model = build_resume_jd_siamese()
model.summary()

Model: "Siamese_Resume_JD_Matcher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_resume        │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_jd            │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 140, 128)  │  1,920,000 │ input_resume[0][… │
│ (Embedding)         │                   │            │ input_jd[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 140)       │          0 │ input_resume[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 140)       │          0 │ input_jd[0][0]    │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm       │ (None, 140, 128)  │     98,816 │ shared_embedding… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0… │
│                     │                   │            │ shared_embedding… │
│                     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_global_max_… │ (None, 128)       │          0 │ shared_bilstm[0]… │
│ (GlobalMaxPooling1… │                   │            │ shared_bilstm[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ absolute_difference │ (None, 128)       │          0 │ shared_global_ma… │
│ (Lambda)            │                   │            │ shared_global_ma… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elementwise_product │ (None, 128)       │          0 │ shared_global_ma… │
│ (Lambda)            │                   │            │ shared_global_ma… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_concatenat… │ (None, 512)       │          0 │ shared_global_ma… │
│ (Concatenate)       │                   │            │ shared_global_ma… │
│                     │                   │            │ absolute_differe… │
│                     │                   │            │ elementwise_prod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     65,664 │ feature_concaten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ match_score (Dense) │ (None, 3)         │        195 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,092,931 (7.98 MB)

 Trainable params: 2,092,931 (7.98 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# CELL 4: Data Loading and Stratified Splitting
# TODO: Replace with your actual dataset filename and column names!
# df = pd.read_csv('your_dataset.csv')

# --- REMOVE THIS BLOCK ONCE YOU LOAD YOUR REAL CSV ---
# Generating dummy data just so the notebook runs for you
df = pd.DataFrame({
    'resume_text': ["Experienced Python dev"] * 1000,
    'jd_text': ["Looking for Python skills"] * 1000,
    'match_label': np.random.choice([0, 1, 2], size=1000)
})
# ----------------------------------------------------

# Stratified Splitting (70% Train, 15% Val, 15% Test)
X_temp_res, X_test_res, X_temp_jd, X_test_jd, y_temp, y_test = train_test_split(
    df['resume_text'].values, df['jd_text'].values, df['match_label'].values,
    test_size=0.15, stratify=df['match_label'].values, random_state=42
)

val_fraction = 0.15 / 0.85
X_train_res, X_val_res, X_train_jd, X_val_jd, y_train, y_val = train_test_split(
    X_temp_res, X_temp_jd, y_temp,
    test_size=val_fraction, stratify=y_temp, random_state=42
)

# Process text
preprocessor = TextPreprocessor()
preprocessor.fit(X_train_res, X_train_jd)

X_train_res_seq = preprocessor.transform(X_train_res)
X_train_jd_seq = preprocessor.transform(X_train_jd)
X_val_res_seq = preprocessor.transform(X_val_res)
X_val_jd_seq = preprocessor.transform(X_val_jd)
X_test_res_seq = preprocessor.transform(X_test_res)
X_test_jd_seq = preprocessor.transform(X_test_jd)

# Calculate balanced class weights
unique_classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
class_weights = dict(zip(unique_classes, weights))
print(f"Class Weights: {class_weights}")

Tokenizer fitted. Actual vocabulary size: 7
Class Weights: {np.int64(0): np.float64(1.023391812865497), np.int64(1): np.float64(1.0101010101010102), np.int64(2): np.float64(0.9681881051175657)}


In [16]:
# CELL 5: Compile, Train, and Evaluate
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

print("Starting training...")
history = model.fit(
    x=[X_train_res_seq, X_train_jd_seq],
    y=y_train,
    validation_data=([X_val_res_seq, X_val_jd_seq], y_val),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop]
)

# Final Evaluation
test_loss, test_acc = model.evaluate([X_test_res_seq, X_test_jd_seq], y_test, verbose=0)
print(f"Final Test Accuracy: {test_acc:.4f}")

# Save Artifacts
model.save('siamese_resume_jd_matcher.keras')
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(preprocessor.tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Model and Tokenizer saved successfully!")

Starting training...
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'shared_global_max_pooling' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


11/11 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.3443 - loss: 1.0988 - val_accuracy: 0.3400 - val_loss: 1.0985
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 782ms/step - accuracy: 0.3429 - loss: 1.0984 - val_accuracy: 0.3400 - val_loss: 1.0985
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 848ms/step - accuracy: 0.3514 - loss: 1.0978 - val_accuracy: 0.3267 - val_loss: 1.0987
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 951ms/step - accuracy: 0.3043 - loss: 1.1009 - val_accuracy: 0.3267 - val_loss: 1.0987
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 874ms/step - accuracy: 0.3314 - loss: 1.0992 - val_accuracy: 0.3267 - val_loss: 1.0987
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 782ms/step - accuracy: 0.3114 - loss: 1.0988 - val_accuracy: 0.3333 - val_loss: 1.0985
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 930ms/step - accuracy: 0.3029 - loss: 1.0995 - val_accuracy: 0.3333 - val_loss: 1.0987
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 954ms/step - accuracy: 0.3114 - loss: 1.0995 - val_accuracy: 0.3267 - v